# AI-Assisted Computing with Large Language Models

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/arvidl/medAI-hands-on/blob/main/notebooks/03_llm_assisted_computing.ipynb)

This notebook demonstrates how to leverage Large Language Models (LLMs) for medical text analysis and research assistance.

**Chapter 22**: Arvid Lundervold *Artificial Intelligence and Computational Medicine: A Hands-on Approach*  
Section 5 - AI-Assisted Computing with Large Language Models

version: 2026-07-19

---

## Learning Objectives

By the end of this notebook, you will be able to:

1. **Understand** LLM capabilities and limitations in medical contexts
2. **Use** Hugging Face transformers for text processing
3. **Implement** summarization, zero-shot classification, biomedical NER, and extractive QA
4. **Generate** analysis code with a small open instruct model (and review it)
5. **Apply** RAG and chain-of-thought prompting to reduce hallucination risk
6. **Evaluate** when to use LLMs vs. traditional NLP approaches

---

## Prerequisites

- Basic Python knowledge
- Some understanding of NLP concepts
- **Runtime**: ~30–45 minutes to read and run (first run downloads models)

<details>
<summary>🔧 <b>Technical Requirements</b></summary>

- **Python 3.8+** with PyTorch 2.0+ and Transformers ≥4.35 (v5 supported)
- **GPU recommended**: Speeds up model inference significantly
- **Memory**: ~8GB RAM minimum for smaller models
- **Storage**: ~3–4GB for model downloads (BART, NER, instruct model; cached after first use)
- **Network**: Required for downloading models from Hugging Face Hub
</details>

<details>
<summary>⚠️ <b>Important Notes on Medical LLM Use</b></summary>

- **LLM outputs require human validation** — never use as authoritative medical advice
- **Never input Protected Health Information (PHI)** into cloud-based models without appropriate safeguards
- **Generated code should always be reviewed** before execution in clinical contexts
- For privacy-sensitive applications, consider **local/on-premise models** (Llama, Mistral, etc.)
</details>


---

## Background and Rationale

Medical knowledge is largely **unstructured text** (notes, literature, reports, guidelines). This notebook shows how open Hugging Face models can help with research workflows—summarization, classification, NER, QA, code drafting, and grounded generation—while keeping human validation central.

<details>
<summary>📖 <b>Clinical context: documentation burden</b></summary>

Physicians often spend substantial time on EHR documentation. LLMs can draft summaries and assist retrieval, but errors in medical text can have serious consequences—outputs are drafts for expert review, not autonomous care.
</details>

### Connection to Notebooks 01–02

```
Notebook 01 (Imaging) → Notebook 02 (Multimodal features) → Notebook 03 (Text / LLMs)
```

| Notebook | Data | Challenge | Approach |
|----------|------|-----------|----------|
| **01** | 3D MRI | Voxel labels | 3D U-Net |
| **02** | Features + clinical | Fusion / missing modalities | Adaptive α,β + PSN/GNN |
| **03** | Clinical / scientific text | Semantics & hallucination | Transformers + RAG |


---

### Maturity framing (read this first)

Two tables appear below on purpose:

1. **Cautious snapshot (≈2025 literature)** — slower, more conservative readiness for *clinical* use.  
2. **Prospective 2026 matrix (next cell)** — aligns with the chapter’s early-2026 assessment of tooling maturity.

They are **not contradictions**: research benchmarks and production tooling move faster than unsupervised clinical autonomy. Human oversight remains required in both views.

#### Cautious snapshot (≈2025 evidence)

| Application | Example | Maturity (cautious) | Evidence sketch |
|-------------|---------|---------------------|-----------------|
| **Literature review** | Screening / summarizing papers | Emerging–operational assist | Tools in active use; humans still lead full SRs |
| **Code assistance** | Analysis scripts | Maturing / widely used | High developer adoption; review before run |
| **Report drafting** | Radiology draft reports | Research–supervised pilots | Vision-LLMs still error-prone without oversight |
| **Clinical Q&A** | Patient / clinician questions | Research–supervised | Few fully unsupervised deployments |
| **Diagnostic support** | Differentials / image quizzes | Research | Benchmark gains ≠ unsupervised diagnosis |

<details>
<summary>📚 <b>Supporting literature (cautious view)</b></summary>

- Gou et al. (2025), *Sci Rep* — humans vs LLMs on systematic review tasks  
- Huang et al. (2025), ResearchCodeBench — novel research code is hard  
- Liu et al. (2025), *npj Digit Med* — vision-LLM neuroradiology accuracy gaps  
- Benary et al. (2025), *Front Digit Health* — real-world clinical LLM workflows  
- Horiuchi et al. (2025), *Front Radiol* — GPT differential accuracy meta-analysis  

</details>

#### Critical limitations (always)

1. **Hallucinations** — mitigate with RAG, CoT, citations, expert review  
2. **Knowledge cutoff** — refresh with retrieval / current sources  
3. **Privacy** — never send PHI to unsecured cloud endpoints; prefer local/on-prem for sensitive data (see chapter: HIPAA/GDPR/PIPEDA, quantization, hybrid)  
4. **Bias & liability** — audit; clinician retains responsibility (EU AI Act / FDA AI/ML guidance)

#### Reinforcement learning for report quality (UniRG)

Hallucination risk in generative radiology reporting is being attacked with **clinical reward signals** (e.g., Microsoft **UniRG**): penalize invented findings, reward clinically correct statements—unlike fluency-only next-token training.

<img src="https://raw.githubusercontent.com/arvidl/medAI-hands-on/main/assets/unirg_RL_framework_2026.png" alt="UniRG Framework" width="700">

*UniRG-style multimodal RL aligns report generation with clinical correctness (see Liu/Zhang et al., UniRG preprint). Discussed in the chapter; not re-implemented in this notebook.*


### Prospective: 2026 maturity matrix (chapter-aligned)

This table matches the chapter’s **early-2026 tooling assessment**. Status labels describe *workflow support*, not unsupervised clinical autonomy.

| Application | 2026 status (chapter) | Key evidence (carefully scoped) |
|-------------|----------------------|----------------------------------|
| **Literature Review** | Operational support | ~85% screening assist / ~40% time reduction reported for AI-assisted SR workflows; PRISMA-trAIce emerging |
| **Code Assistance** | Ubiquitous | High developer adoption; strong agentic coding benchmarks (e.g., SWE-bench Verified) — still review before use |
| **Report Drafting** | Advancing under supervision | Ambient scribes / RL report frameworks (e.g., UniRG) target hallucination reduction. **Note:** “>1,000 FDA AI authorizations” are mostly **imaging devices**, not approvals of generative report writers |
| **Clinical Q&A** | Deployed with supervision | High draft rates in some health systems; human-in-the-loop required for patient-facing text |
| **Diagnostic Support** | Supervised / research | Strong scores on **selected quizzes/benchmarks** (e.g., NEJM Image Challenge MCQs). That is **not** the same as unsupervised bedside diagnosis |

**Takeaway for this notebook:** open models below are for **research assistance and mechanism demos**. Prefer supervised integration; validate every clinical-facing output.


---

### What This Notebook Covers

Practical LLM workflows for biomedical research using **open-source Hugging Face models** (no proprietary API required):

1. **Text summarization** — abstractive condensation with BART
2. **Zero-shot classification** — label medical abstracts without task-specific training
3. **Biomedical NER** — transformer token-classification for clinical/scientific entities
4. **Code generation** — small instruct model drafts analysis helpers (always review before use)
5. **Extractive question answering** — span answers from abstracts (DistilBERT/SQuAD)
6. **Retrieval-Augmented Generation (RAG (teaching corpus))** — ChromaDB + embeddings + grounded generation
7. **Hallucination mitigations** — RAG grounding, chain-of-thought prompting, output variability

Privacy note: these demos run locally (or in Colab) on public sample text — never paste PHI into cloud endpoints without appropriate agreements.

---


## 1. Environment Setup

In [1]:
# Check if running in Google Colab
import sys
IN_COLAB = 'google.colab' in sys.modules

if IN_COLAB:
    print("Running in Google Colab - installing dependencies...")
    %pip install -q transformers accelerate sentencepiece
    %pip install -q chromadb sentence-transformers
    print("✓ Dependencies installed")
else:
    # For local execution, add parent directory to path
    sys.path.insert(0, '..')

print(f"Python version: {sys.version}")

Python version: 3.11.15 (main, Apr 14 2026, 14:28:36) [Clang 22.1.3 ]


In [ ]:
# Core imports
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
import warnings
import gc
warnings.filterwarnings('ignore')

# Deep learning and NLP
import torch
from transformers import pipeline, AutoTokenizer, AutoModelForSeq2SeqLM
from transformers import AutoModelForCausalLM, set_seed as hf_set_seed

# ============================================================================
# IMPORT STRATEGY: Use src/ modules locally, define inline for Colab
# ============================================================================
if IN_COLAB:
    # Define get_device inline for Colab
    def get_device(force_cpu_for_3d=False):
        """Get the best available device (CUDA, MPS, or CPU)."""
        if torch.cuda.is_available():
            device = torch.device("cuda")
            print(f"Using CUDA: {torch.cuda.get_device_name(0)}")
        elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
            if force_cpu_for_3d:
                device = torch.device("cpu")
                print("Using CPU (MPS available but 3D ops not fully supported)")
            else:
                device = torch.device("mps")
                print("Using Apple Silicon MPS (with CPU fallback for unsupported ops)")
        else:
            device = torch.device("cpu")
            print("Using CPU")
        return device
else:
    # For local execution, import from src modules
    from src.data_utils import get_device

# ============================================================================
# REPRODUCIBILITY: Set all random seeds for reproducible results
# ============================================================================
SEED = 42

def set_seed(seed: int = 42):
    """Set all random seeds for reproducibility."""
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed(seed)
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False
    # Also set Hugging Face transformers seed
    hf_set_seed(seed)

set_seed(SEED)
print(f"Random seed set to: {SEED}")
print(f"PyTorch version: {torch.__version__}")

def free_globals(*names):
    """Drop large model objects and free GPU cache between stages."""
    g = globals()
    for n in names:
        if n in g:
            del g[n]
    gc.collect()
    if torch.cuda.is_available():
        torch.cuda.empty_cache()


In [3]:
# Get device
device = get_device()

# Check for TPU (Colab TPU runtime)
TPU_AVAILABLE = False
try:
    import torch_xla.core.xla_model as xm
    TPU_AVAILABLE = True
    print("⚠️ TPU detected but Hugging Face pipelines have limited TPU support.")
    print("   Recommend: Use Colab GPU runtime (Runtime → Change runtime type → GPU)")
except ImportError:
    pass

# Helper function for Hugging Face pipeline device selection
# Pipelines use: 0 for CUDA, "mps" for Apple Silicon, -1 for CPU
def get_pipeline_device():
    """Get device specification for Hugging Face pipelines."""
    if torch.cuda.is_available():
        return 0  # CUDA GPU 0 (Colab GPU, NVIDIA)
    elif hasattr(torch.backends, 'mps') and torch.backends.mps.is_available():
        return "mps"  # Apple Silicon
    else:
        return -1  # CPU (or TPU fallback - pipelines don't support TPU directly)

PIPELINE_DEVICE = get_pipeline_device()

# Print device info
print(f"\nDevice configuration:")
print(f"  PyTorch device: {device}")
print(f"  Pipeline device: {PIPELINE_DEVICE}")
if IN_COLAB:
    if torch.cuda.is_available():
        print(f"  ✓ Colab GPU runtime detected")
    else:
        print(f"  ⚠️ No GPU - for faster inference, use: Runtime → Change runtime type → GPU")

# For LLMs, check GPU memory
if torch.cuda.is_available():
    gpu_mem = torch.cuda.get_device_properties(0).total_memory / 1e9
    print(f"  GPU memory: {gpu_mem:.1f} GB")
    if gpu_mem < 8:
        print("  ⚠️ Limited GPU memory - using smaller models")

Using CUDA: NVIDIA RTX A5000 Laptop GPU

Device configuration:
  PyTorch device: cuda
  Pipeline device: 0
  GPU memory: 16.8 GB


## 2. Sample Medical Texts

We'll use sample medical abstracts and clinical text snippets for demonstration. In practice, you would use your own research documents or publicly available medical literature.

In [4]:
# Sample medical abstracts for demonstration
SAMPLE_ABSTRACTS = [
    {
        "title": "Deep Learning for Brain Tumor Segmentation",
        "abstract": """Background: Accurate segmentation of brain tumors from MRI scans is crucial for 
        diagnosis, treatment planning, and monitoring. Deep learning methods have shown promising 
        results in automating this task. Methods: We developed a 3D U-Net architecture trained on 
        the BraTS dataset consisting of 1,251 multiparametric MRI scans. The network was trained 
        using Dice loss with data augmentation including random rotations, flips, and intensity 
        variations. Results: Our method achieved a mean Dice score of 0.89 for whole tumor, 0.83 
        for tumor core, and 0.79 for enhancing tumor on the validation set. The model processes 
        a complete brain volume in under 2 seconds on a standard GPU. Conclusions: Deep learning 
        enables accurate, fast, and reproducible brain tumor segmentation that could support 
        clinical workflows and reduce inter-observer variability."""
    },
    {
        "title": "Multimodal Learning in Precision Medicine",
        "abstract": """Precision medicine requires integration of diverse data modalities including 
        imaging, genomics, and clinical variables. We present a multimodal deep learning framework 
        that combines radiomics features extracted from CT scans with genomic profiles and clinical 
        data to predict treatment response in non-small cell lung cancer patients. Using a cohort 
        of 422 patients, our attention-based fusion model achieved an AUC of 0.82 compared to 
        0.71 for imaging-only and 0.68 for genomics-only models. The attention mechanism revealed 
        that imaging features dominated predictions for early-stage tumors while genomic features 
        were more important for advanced disease. This work demonstrates the value of multimodal 
        integration for personalized treatment selection."""
    },
    {
        "title": "Large Language Models in Clinical Documentation",
        "abstract": """Large language models (LLMs) have potential applications in clinical 
        documentation, including summarization and information extraction. We evaluated GPT-4 
        and clinical-domain fine-tuned models on summarizing discharge notes from 500 hospital 
        admissions. Human evaluators rated summaries for accuracy, completeness, and clinical 
        relevance. Fine-tuned models achieved 87% accuracy compared to 79% for general-purpose 
        LLMs. However, both exhibited occasional hallucinations requiring human oversight. 
        We recommend LLMs as assistive tools with mandatory clinician review rather than 
        autonomous documentation systems."""
    }
]

print(f"Loaded {len(SAMPLE_ABSTRACTS)} sample abstracts")
for i, item in enumerate(SAMPLE_ABSTRACTS):
    print(f"  {i+1}. {item['title']}")

Loaded 3 sample abstracts
  1. Deep Learning for Brain Tumor Segmentation
  2. Multimodal Learning in Precision Medicine
  3. Large Language Models in Clinical Documentation


## 3. Text Summarization

### Clinical Motivation

Medical professionals face an overwhelming volume of literature—over **5 million new articles** are published annually. Text summarization helps clinicians and researchers:

- **Rapid literature triage**: Quickly assess relevance of papers during systematic reviews
- **Patient record summarization**: Condense lengthy patient histories into actionable summaries
- **Report generation**: Create concise summaries of radiology or pathology findings

<details>
<summary>📖 <b>How Abstractive Summarization Works</b></summary>

Unlike **extractive summarization** (selecting key sentences verbatim), **abstractive summarization** generates new text that captures the essence of the original. Models like BART and T5 are trained on document-summary pairs to learn this mapping.

**BART (Bidirectional and Auto-Regressive Transformers)** uses:
1. A corrupted input (noising) during training
2. An encoder-decoder architecture to reconstruct the original
3. Fine-tuning on summarization datasets (CNN/DailyMail, XSum)

For medical text, models can be further fine-tuned on biomedical corpora (PubMed, clinical notes) to improve domain-specific performance.
</details>

---

We use a pre-trained **BART-large-CNN** model to condense medical abstracts. This demonstrates practical summarization for literature review and rapid content assessment.

In [5]:
# Load summarization model
# Using BART-large-CNN for text summarization
# Note: Newer transformers versions may not have "summarization" as a pipeline task,
# so we load the model and tokenizer directly for maximum compatibility
print("Loading summarization model...")

from transformers import BartForConditionalGeneration, BartTokenizer

summarizer_model = BartForConditionalGeneration.from_pretrained("facebook/bart-large-cnn")
summarizer_tokenizer = BartTokenizer.from_pretrained("facebook/bart-large-cnn")

# Move model to appropriate device
summarizer_model = summarizer_model.to(device)
summarizer_model.eval()

print(f"Model loaded on {device}!")

Loading summarization model...


[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

Model loaded on cuda!


In [6]:
def summarize_text(text, max_length=100, min_length=30):
    """Summarize a text using the loaded BART model."""
    # Clean up whitespace
    text = " ".join(text.split())
    
    # Tokenize input
    inputs = summarizer_tokenizer(
        text, 
        return_tensors="pt", 
        max_length=1024, 
        truncation=True
    ).to(device)
    
    # Generate summary
    with torch.no_grad():
        summary_ids = summarizer_model.generate(
            inputs["input_ids"],
            max_length=max_length,
            min_length=min_length,
            num_beams=4,
            length_penalty=2.0,
            early_stopping=True
        )
    
    # Decode and return
    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    return summary

In [7]:
# Summarize the sample abstracts
print("=" * 60)
print("MEDICAL ABSTRACT SUMMARIZATION")
print("=" * 60)

for item in SAMPLE_ABSTRACTS:
    print(f"\n📄 {item['title']}")
    print("-" * 40)
    
    original = item['abstract']
    summary = summarize_text(original)
    
    print(f"Original ({len(original.split())} words):")
    print(f"  {original[:200]}...")
    print(f"\nSummary ({len(summary.split())} words):")
    print(f"  {summary}")
    print()

MEDICAL ABSTRACT SUMMARIZATION

📄 Deep Learning for Brain Tumor Segmentation
----------------------------------------
Original (124 words):
  Background: Accurate segmentation of brain tumors from MRI scans is crucial for 
        diagnosis, treatment planning, and monitoring. Deep learning methods have shown promising 
        results in a...

Summary (34 words):
  We developed a 3D U-Net architecture trained on the BraTS dataset consisting of 1,251 multiparametric MRI scans. The network was trained using Dice loss with data augmentation including random rotations, flips, and intensity variations.


📄 Multimodal Learning in Precision Medicine
----------------------------------------
Original (103 words):
  Precision medicine requires integration of diverse data modalities including 
        imaging, genomics, and clinical variables. We present a multimodal deep learning framework 
        that combines ...

Summary (46 words):
  We present a multimodal deep learning framework that com

## 4. Text Classification

### Clinical Motivation

Automated text classification enables systematic organization of medical documents at scale:

- **Literature categorization**: Automatically tag papers by topic, methodology, or disease area
- **Clinical note classification**: Identify note types (progress notes, discharge summaries, consults)
- **Triage and routing**: Route patient messages to appropriate specialists
- **Sentiment/urgency detection**: Flag high-priority communications

<details>
<summary>📖 <b>Zero-Shot Classification Explained</b></summary>

Traditional classifiers require labeled training data for each category. **Zero-shot classification** can classify text into *arbitrary categories* without task-specific training:

1. **Natural Language Inference (NLI)**: The model is trained to determine if a "hypothesis" is entailed by a "premise"
2. **Template matching**: Each category becomes a hypothesis (e.g., "This text is about {category}")
3. **Probability scoring**: The model scores how likely each hypothesis is given the input text

**Example**:
- Input: "Patient presents with chest pain and shortness of breath"
- Candidate labels: ["cardiology", "pulmonology", "neurology"]
- Model evaluates: "This text is about cardiology" → High probability

This approach is powerful for exploratory analysis when predefined categories don't exist.
</details>

---

We use **zero-shot classification** with BART-MNLI to categorize medical texts without task-specific training—useful for organizing research literature or clinical notes.

In [8]:
# Load zero-shot classification pipeline
print("Loading classification model...")
classifier = pipeline(
    "zero-shot-classification",
    model="facebook/bart-large-mnli",
    device=PIPELINE_DEVICE  # Supports CUDA, MPS, or CPU
)
print("Model loaded!")

Loading classification model...


Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

Model loaded!


In [9]:
# Define candidate labels for classification
candidate_labels = [
    "medical imaging",
    "genomics and molecular biology",
    "clinical trials",
    "drug discovery",
    "health informatics",
    "natural language processing"
]

# Classify each abstract
print("=" * 60)
print("ZERO-SHOT TEXT CLASSIFICATION")
print("=" * 60)

for item in SAMPLE_ABSTRACTS:
    result = classifier(
        item['abstract'][:500],  # Limit length
        candidate_labels,
        multi_label=True
    )
    
    print(f"\n📄 {item['title']}")
    print("-" * 40)
    print("Top categories:")
    for label, score in zip(result['labels'][:3], result['scores'][:3]):
        print(f"  • {label}: {score:.2%}")

ZERO-SHOT TEXT CLASSIFICATION

📄 Deep Learning for Brain Tumor Segmentation
----------------------------------------
Top categories:
  • medical imaging: 98.36%
  • health informatics: 77.14%
  • clinical trials: 2.53%

📄 Multimodal Learning in Precision Medicine
----------------------------------------
Top categories:
  • medical imaging: 96.84%
  • genomics and molecular biology: 17.70%
  • health informatics: 10.38%

📄 Large Language Models in Clinical Documentation
----------------------------------------
Top categories:
  • natural language processing: 38.36%
  • health informatics: 13.49%
  • clinical trials: 3.00%


In [ ]:
# Free zero-shot classifier weights before loading NER (keeps BART summarizer for §9 variability demo)
free_globals("classifier")
print("✓ Released zero-shot classifier from memory")


## 5. Named Entity Recognition

### Clinical Motivation

**Named Entity Recognition (NER)** extracts structured information from unstructured clinical text—a critical step in building clinical NLP pipelines:

- **Medication extraction**: Identify drug names, dosages, frequencies from prescription notes
- **Diagnosis coding**: Extract disease mentions for ICD coding
- **Adverse event detection**: Find mentions of symptoms, side effects, complications
- **Temporal reasoning**: Extract dates, durations, and temporal relationships

<details>
<summary>📖 <b>Medical NER: Challenges and Approaches</b></summary>

Medical NER is more challenging than general-domain NER due to:

| Challenge | Example |
|-----------|---------|
| **Abbreviations** | "pt c/o SOB" = "patient complains of shortness of breath" |
| **Synonyms** | "heart attack" = "MI" = "myocardial infarction" |
| **Negation** | "no evidence of pneumonia" (negative finding) |
| **Uncertainty** | "possible malignancy" vs "confirmed malignancy" |
| **Context dependence** | "discharge" (hospital discharge vs wound discharge) |

**Specialized biomedical NER models** (BioBERT, ClinicalBERT, PubMedBERT, domain-tuned token classifiers) are preferred for production. Complementary tools include **scispaCy**, **MedCAT**, and **BERN2**.
</details>

---

We run a **transformer token-classification** model fine-tuned for biomedical entities
(`d4data/biomedical-ner-all`). A regex/keyword baseline is kept only as an automatic fallback if the model cannot be loaded.


In [10]:
# Biomedical NER via Hugging Face token-classification
# (transformers v5: use task "token-classification" / alias "ner")
print("Loading biomedical NER model...")

NER_MODEL_NAME = "d4data/biomedical-ner-all"
NER_AVAILABLE = False
ner_pipeline = None

try:
    ner_pipeline = pipeline(
        "token-classification",
        model=NER_MODEL_NAME,
        aggregation_strategy="simple",
        device=PIPELINE_DEVICE,
    )
    NER_AVAILABLE = True
    print(f"✓ NER ready: {NER_MODEL_NAME}")
except Exception as e:
    print(f"⚠️ Could not load NER model ({type(e).__name__}: {e})")
    print("  Falling back to a lightweight keyword/regex extractor for this session.")


def extract_medical_entities(text, score_threshold=0.5):
    """Extract biomedical entities; prefer transformer NER when available."""
    if NER_AVAILABLE:
        raw = ner_pipeline(text)
        entities = {}
        for item in raw:
            label = item.get("entity_group") or item.get("entity")
            score = float(item.get("score", 0.0))
            word = (item.get("word") or "").replace("##", "").strip()
            if score < score_threshold or not word:
                continue
            bucket = entities.setdefault(label, [])
            if word not in bucket:
                bucket.append(word)
        return entities

    # Fallback: transparent pattern/keyword baseline (not production NER)
    import re
    entities = {"metrics": [], "methods": [], "diseases": []}
    for pattern in [
        r"(\d+\.?\d*\s*%)",
        r"(Dice\s*(?:score\s*)?(?:of\s*)?\d+\.?\d*)",
        r"(\d+\s*(?:patients?|subjects?|samples?))",
    ]:
        entities["metrics"].extend(re.findall(pattern, text, flags=re.IGNORECASE))
    for method in ["U-Net", "deep learning", "MRI", "radiomics", "multimodal"]:
        if method.lower() in text.lower():
            entities["methods"].append(method)
    for disease in ["tumor", "cancer", "glioma", "glioblastoma", "lung cancer", "brain tumor"]:
        if disease.lower() in text.lower():
            entities["diseases"].append(disease)
    return entities


Loading biomedical NER model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

✓ NER ready: d4data/biomedical-ner-all


In [11]:
# Extract entities from sample abstracts
print("=" * 60)
print("NAMED ENTITY EXTRACTION")
print("=" * 60)
print(
    f"Backend: {'transformer NER (' + NER_MODEL_NAME + ')' if NER_AVAILABLE else 'regex/keyword fallback'}"
)

for item in SAMPLE_ABSTRACTS:
    entities = extract_medical_entities(item["abstract"])
    print(f"\n📄 {item['title']}")
    print("-" * 40)
    if not entities:
        print("  (no entities above threshold)")
        continue
    for entity_type, values in entities.items():
        unique_values = list(dict.fromkeys(values))[:8]
        print(f"  {entity_type}: {', '.join(unique_values)}")


NAMED ENTITY EXTRACTION
Backend: transformer NER (d4data/biomedical-ner-all)

📄 Deep Learning for Brain Tumor Segmentation
----------------------------------------
  Biological_structure: brain, brain volume
  Coreference: tumors, brat, tumor
  Diagnostic_procedure: mri, deep, 3d, net architecture
  Detailed_description: u -
  Lab_value: 0, 89, 83

📄 Multimodal Learning in Precision Medicine
----------------------------------------
  Detailed_description: precision, multi
  Diagnostic_procedure: mics, ct, fusion model, imaging, gen
  Biological_structure: lung
  Lab_value: auc, 0. 82, 71, 0, 68

📄 Large Language Models in Clinical Documentation
----------------------------------------
  Detailed_description: models, clinical, domain
  Disease_disorder: ll, ms
  Diagnostic_procedure: gpt - 4, -, models, human eva, ators, fine, tuned models
  Lab_value: 87 % accuracy, %
  Sign_symptom: hall, uc
  Coreference: ms


In [ ]:
# Free NER pipeline before loading the instruct generator / QA model
free_globals("ner_pipeline", "extract_medical_entities")
print("✓ Released NER model from memory")


## 6. Code Generation Assistance

### Accelerating Biomedical Research

LLMs have transformed software development, with **84% of developers** now using AI coding assistants (Stack Overflow, 2025). For biomedical researchers, this means:

- **Faster prototyping**: Generate boilerplate for survival analysis, image preprocessing, statistical tests
- **Learning tool**: Understand unfamiliar libraries by asking for annotated examples
- **Debugging assistance**: Explain error messages and suggest fixes
- **Documentation**: Generate docstrings, comments, and README files

<details>
<summary>📖 <b>Code Generation: Best Practices</b></summary>

**What LLMs do well:**
- Boilerplate and template code
- Standard library usage (pandas, numpy, sklearn)
- Well-documented APIs (matplotlib, seaborn)
- Common patterns (data loading, preprocessing, visualization)

**What requires caution:**
- Novel algorithms or cutting-edge research code (~37% accuracy on research-level tasks)
- Security-critical code (authentication, encryption)
- Performance-critical optimizations
- Domain-specific statistical methods (verify assumptions!)

**Recommended workflow:**
1. **Prompt clearly**: Specify language, libraries, input/output formats
2. **Request explanations**: Ask the model to explain its reasoning
3. **Review thoroughly**: Check logic, edge cases, and assumptions
4. **Test rigorously**: Unit tests, integration tests, validation on known data
5. **Iterate**: Refine based on test results
</details>

---

We load a **small open instruct model** (`SmolLM2-360M-Instruct`) via the `text-generation` pipeline and ask it to draft a Dice helper. We then show a curated reference template—the kind of artifact you keep **after** human review. The same generator is reused later for RAG answers and chain-of-thought prompts.


In [12]:
# Shared small instruct model for code generation, RAG answers, and CoT demos
print("Loading instruct generator (SmolLM2-360M-Instruct)...")

GENERATOR_MODEL = "HuggingFaceTB/SmolLM2-360M-Instruct"
GENERATOR_AVAILABLE = False
chat_pipeline = None

try:
    chat_pipeline = pipeline(
        "text-generation",
        model=GENERATOR_MODEL,
        device=PIPELINE_DEVICE,
    )
    GENERATOR_AVAILABLE = True
    print(f"✓ Generator ready: {GENERATOR_MODEL}")
except Exception as e:
    print(f"⚠️ Could not load generator ({type(e).__name__}: {e})")
    print("  Code-gen / RAG-answer / CoT cells will show fallbacks.")


def chat_generate(prompt: str, max_new_tokens: int = 220) -> str:
    """Single-turn chat generation; returns assistant text."""
    if not GENERATOR_AVAILABLE:
        return "[generator unavailable]"
    messages = [{"role": "user", "content": prompt}]
    out = chat_pipeline(
        messages,
        max_new_tokens=max_new_tokens,
        do_sample=False,
        return_full_text=False,
    )
    gen = out[0]["generated_text"]
    if isinstance(gen, list):
        return gen[-1].get("content", str(gen[-1]))
    return str(gen)


# --- Live code generation ---
code_prompt = """Write a short, correct Python function `dice_score(y_true, y_pred)` for binary NumPy arrays.
Requirements:
- import numpy as np
- handle the empty-union edge case safely
- include a one-line docstring
Output ONLY the code (no prose)."""

print("=" * 60)
print("LLM CODE GENERATION")
print("=" * 60)
if GENERATOR_AVAILABLE:
    generated_code = chat_generate(code_prompt, max_new_tokens=220)
    print(generated_code)
    print("\n⚠️ Review before use: check edge cases, types, and numerical stability.")
else:
    generated_code = None
    print("Skipped — generator not available.")


Loading instruct generator (SmolLM2-360M-Instruct)...


Loading weights:   0%|          | 0/290 [00:00<?, ?it/s]

[transformers] Passing `generation_config` together with generation-related arguments=({'do_sample', 'max_new_tokens'}) is deprecated and will be removed in future versions. Please pass either a `generation_config` object OR all generation parameters explicitly, but not both.
[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


✓ Generator ready: HuggingFaceTB/SmolLM2-360M-Instruct
LLM CODE GENERATION


[transformers] Ignoring clean_up_tokenization_spaces=True for BPE tokenizer GPT2Tokenizer. The clean_up_tokenization post-processing step is designed for WordPiece tokenizers and is destructive for BPE (it strips spaces before punctuation). Set clean_up_tokenization_spaces=False to suppress this warning, or set clean_up_tokenization_spaces_for_bpe_even_though_it_will_corrupt_output=True to force cleanup anyway.


```python
import numpy as np

def dice_score(y_true, y_pred):
    """
    Calculate the dice score for binary arrays.

    Args:
        y_true (np.ndarray): True labels.
        y_pred (np.ndarray): Predicted labels.

    Returns:
        float: Dice score.
    """
    if y_true.size == 0 or y_pred.size == 0:
        return 0.0
    return 2 * (1 - np.sum(np.logical_and(y_true, y_pred))) / (y_true.size + y_pred.size)
```

⚠️ Review before use: check edge cases, types, and numerical stability.


In [13]:
# Curated reference templates (what you keep AFTER reviewing LLM drafts)
# These are not model outputs — they illustrate a human-validated baseline.

CODE_TEMPLATES = {
    "dice_score": '''
import numpy as np

def dice_score(y_true, y_pred, eps=1e-8):
    """Dice similarity for binary arrays (or binarized inputs)."""
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    intersection = np.logical_and(y_true, y_pred).sum()
    denom = y_true.sum() + y_pred.sum()
    if denom == 0:
        return 1.0
    return float((2.0 * intersection) / (denom + eps))
''',
    "survival_analysis": '''
# Survival Analysis with Kaplan-Meier
import pandas as pd
from lifelines import KaplanMeierFitter
import matplotlib.pyplot as plt

def perform_survival_analysis(data, time_col, event_col, group_col=None):
    """Perform Kaplan-Meier survival analysis."""
    kmf = KaplanMeierFitter()
    if group_col is None:
        kmf.fit(data[time_col], data[event_col])
        kmf.plot_survival_function()
    else:
        for group in data[group_col].unique():
            mask = data[group_col] == group
            kmf.fit(data.loc[mask, time_col], data.loc[mask, event_col], label=str(group))
            kmf.plot_survival_function()
    plt.xlabel("Time")
    plt.ylabel("Survival Probability")
    plt.title("Kaplan-Meier Survival Curves")
    return kmf
''',
}

print("=" * 60)
print("REVIEWED REFERENCE TEMPLATE: dice_score")
print("=" * 60)
print(CODE_TEMPLATES["dice_score"])
print("Compare the LLM draft above with this reviewed version.")
print("Available templates:", ", ".join(CODE_TEMPLATES))


REVIEWED REFERENCE TEMPLATE: dice_score

import numpy as np

def dice_score(y_true, y_pred, eps=1e-8):
    """Dice similarity for binary arrays (or binarized inputs)."""
    y_true = np.asarray(y_true).astype(bool)
    y_pred = np.asarray(y_pred).astype(bool)
    intersection = np.logical_and(y_true, y_pred).sum()
    denom = y_true.sum() + y_pred.sum()
    if denom == 0:
        return 1.0
    return float((2.0 * intersection) / (denom + eps))

Compare the LLM draft above with this reviewed version.
Available templates: dice_score, survival_analysis


## 7. Question Answering

### Extracting Specific Information

**Extractive Question Answering (QA)** locates and extracts answers from a given context—particularly useful for:

- **Literature mining**: Find specific values, methods, or results in papers
- **Clinical queries**: "What medications is this patient on?" from clinical notes
- **Guideline navigation**: Extract specific recommendations from lengthy protocols
- **Research synthesis**: Pull comparable metrics across multiple studies

<details>
<summary>📖 <b>Extractive vs. Generative QA</b></summary>

| Type | How it works | Example |
|------|-------------|---------|
| **Extractive QA** | Highlights a span from the input text | Q: "What was the accuracy?" → A: "**94.2%**" (exact quote from text) |
| **Generative QA** | Generates a new answer based on context | Q: "Summarize the findings" → A: "The study showed improved outcomes..." |

**Extractive QA** is preferred when you need:
- Exact quotes and citations
- Verifiable answers (can point to source)
- Numerical values and statistics

**Generative QA** (using larger LLMs) is better for:
- Synthesis across multiple passages
- Paraphrasing and explanation
- Follow-up reasoning
</details>

---

We use **DistilBERT** fine-tuned on SQuAD to extract specific information from medical abstracts. (As of transformers v5 the dedicated `question-answering` pipeline was removed, so the next cell loads the model and tokenizer directly while keeping extractive span prediction.)


In [14]:
# Load extractive QA model (DistilBERT fine-tuned on SQuAD)
# Note: transformers v5 removed the "question-answering" pipeline task, so we
# load AutoModelForQuestionAnswering + tokenizer directly (same pattern as summarization).
print("Loading QA model...")

from transformers import AutoModelForQuestionAnswering, AutoTokenizer

QA_MODEL_NAME = "distilbert-base-cased-distilled-squad"
qa_tokenizer = AutoTokenizer.from_pretrained(QA_MODEL_NAME)
qa_net = AutoModelForQuestionAnswering.from_pretrained(QA_MODEL_NAME)
qa_net = qa_net.to(device)
qa_net.eval()


def qa_model(question, context, max_length=512):
    """Extractive QA: return the highest-scoring answer span from context."""
    inputs = qa_tokenizer(
        question,
        context,
        return_tensors="pt",
        truncation="only_second",
        max_length=max_length,
    )
    inputs = {k: v.to(device) for k, v in inputs.items()}

    with torch.no_grad():
        outputs = qa_net(**inputs)

    start_logits = outputs.start_logits[0]
    end_logits = outputs.end_logits[0]
    start_idx = int(torch.argmax(start_logits))
    end_idx = int(torch.argmax(end_logits))
    if end_idx < start_idx:
        end_idx = start_idx

    input_ids = inputs["input_ids"][0]
    answer = qa_tokenizer.decode(
        input_ids[start_idx : end_idx + 1], skip_special_tokens=True
    ).strip()
    score = (
        torch.softmax(start_logits, dim=-1)[start_idx]
        * torch.softmax(end_logits, dim=-1)[end_idx]
    ).item()
    return {"answer": answer, "score": score}


print(f"Model loaded on {device}!")


Loading QA model...


Loading weights:   0%|          | 0/102 [00:00<?, ?it/s]

Model loaded on cuda!


In [15]:
# Ask questions about the medical abstracts
context = SAMPLE_ABSTRACTS[0]['abstract']

questions = [
    "What was the mean Dice score for whole tumor?",
    "How long does the model take to process a brain volume?",
    "What dataset was used for training?"
]

print("=" * 60)
print("QUESTION ANSWERING")
print("=" * 60)
print(f"\nContext: {SAMPLE_ABSTRACTS[0]['title']}")
print("-" * 40)

for question in questions:
    result = qa_model(question=question, context=context)
    print(f"\nQ: {question}")
    print(f"A: {result['answer']} (confidence: {result['score']:.2%})")

QUESTION ANSWERING

Context: Deep Learning for Brain Tumor Segmentation
----------------------------------------

Q: What was the mean Dice score for whole tumor?
A: 0. 89 (confidence: 99.63%)

Q: How long does the model take to process a brain volume?
A: under 2 seconds (confidence: 63.50%)

Q: What dataset was used for training?
A: BraTS (confidence: 52.92%)


In [ ]:
# Free extractive QA weights; keep instruct generator for RAG + CoT
free_globals("qa_tokenizer", "qa_model", "answer_question")
print("✓ Released extractive QA model from memory")


## 8. Retrieval-Augmented Generation (RAG)

### Grounding answers in retrieved documents

Base LLMs can **hallucinate**. **RAG** reduces that risk by retrieving relevant passages and conditioning generation on them.

**In this notebook** we use a small **teaching corpus** (a handful of short authored notes about topics from Notebooks 01–03). That is enough to learn the mechanism. A production system would index **verified** guidelines and peer-reviewed literature (as discussed in the chapter)—not this toy set.

**Pipeline**

1. **Index** — embed documents → store in a vector DB  
2. **Retrieve** — embed the query → top-*k* nearest docs  
3. **Generate** — prompt an instruct LLM with context + question  

```
Document corpus  →  MiniLM embeddings  →  ChromaDB (cosine space)
User query       →  embed            →  top-k chunks
                                     →  SmolLM2 (context + question) → answer
```

<details>
<summary>📖 <b>Why RAG for medical AI?</b></summary>

- Grounds answers in inspectable text  
- Supports citations / source checks  
- Updates knowledge by adding documents (no full retrain)  
- Can keep corpora on-prem for privacy  

</details>

**Stack here:** ChromaDB + `all-MiniLM-L6-v2` + the **SmolLM2** instruct model from §6.


In [16]:
# Install RAG dependencies (uncomment if not installed)
# %pip install -q chromadb sentence-transformers

# Import RAG components
try:
    import chromadb
    from chromadb.utils import embedding_functions
    from sentence_transformers import SentenceTransformer
    RAG_AVAILABLE = True
    print("✓ RAG dependencies available")
except ImportError as e:
    print(f"⚠️ RAG dependencies not installed: {e}")
    print("Install with: pip install chromadb sentence-transformers")
    RAG_AVAILABLE = False

✓ RAG dependencies available


In [ ]:
# Teaching corpus for RAG mechanism demo (not a verified literature database).
# Production systems would index PubMed, guidelines, etc.

MEDICAL_DOCUMENTS = [
    {
        "id": "doc1",
        "title": "U-Net Architecture for Medical Image Segmentation",
        "content": """The U-Net architecture was introduced by Ronneberger et al. in 2015 for biomedical 
        image segmentation. It features an encoder-decoder structure with skip connections that 
        concatenate features from the contracting path to the expanding path. This allows the 
        network to combine high-level semantic information with fine-grained spatial details.
        The architecture is particularly effective for medical imaging tasks where precise 
        boundary delineation is critical. U-Net has become the foundation for many medical 
        imaging segmentation models, including nnU-Net which automates architecture decisions."""
    },
    {
        "id": "doc2", 
        "title": "Dice Loss for Imbalanced Segmentation",
        "content": """The Dice loss function, based on the Dice coefficient (also known as F1 score), 
        directly optimizes the overlap between predicted segmentation and ground truth. It is 
        particularly effective for imbalanced datasets where the region of interest (e.g., tumor) 
        occupies a small fraction of the total volume. The Dice loss is defined as:
        L_dice = 1 - (2 * sum(p * g) + epsilon) / (sum(p) + sum(g) + epsilon)
        where p is the predicted probability and g is the ground truth. The epsilon term 
        (typically 1e-5) ensures numerical stability. Combining Dice loss with cross-entropy 
        often provides more stable training than using either loss alone."""
    },
    {
        "id": "doc3",
        "title": "BraTS Challenge and Brain Tumor Segmentation",
        "content": """The Brain Tumor Segmentation (BraTS) Challenge provides a benchmark dataset 
        for evaluating brain tumor segmentation algorithms. The dataset contains multiparametric 
        MRI scans (T1, T1-Gd, T2, FLAIR) with expert annotations of tumor sub-regions: 
        enhancing tumor (ET), tumor core (TC), and whole tumor (WT). State-of-the-art methods 
        achieve Dice scores above 0.90 for whole tumor segmentation. The challenge has driven 
        significant advances in medical image segmentation since its inception in 2012."""
    },
    {
        "id": "doc4",
        "title": "Multimodal Data Fusion in Medical AI",
        "content": """Multimodal data integration combines heterogeneous data sources—imaging, 
        genomics, clinical variables—to improve predictive accuracy. Attention-based fusion 
        mechanisms learn optimal weights for each modality, enabling the model to emphasize 
        the most informative data source for each patient. This approach is particularly 
        valuable when different modalities capture complementary aspects of disease. For 
        example, imaging may reveal spatial tumor characteristics while genomics indicates 
        molecular subtypes. Patient similarity networks constructed from fused embeddings 
        can reveal clinically meaningful patient subgroups."""
    },
    {
        "id": "doc5",
        "title": "Hallucination Mitigation in Medical LLMs",
        "content": """Large language models can generate plausible but factually incorrect 
        information (hallucination), which is particularly dangerous in medical contexts. 
        Mitigation strategies include: (1) Retrieval-Augmented Generation (RAG) to ground 
        responses in verified documents; (2) Chain-of-thought prompting to make reasoning 
        explicit and verifiable; (3) Reinforcement learning from clinical feedback, as 
        demonstrated by Microsoft's UniRG framework which uses clinical reward signals to 
        reduce hallucinations in radiology reports; (4) Calibrated confidence scores to 
        flag uncertain outputs for human review. Human-in-the-loop validation remains 
        essential for any clinical application."""
    }
]

print(f"Created corpus with {len(MEDICAL_DOCUMENTS)} documents:")
for doc in MEDICAL_DOCUMENTS:
    print(f"  • {doc['title']}")

In [ ]:
if RAG_AVAILABLE:
    print("Loading embedding model...")
    embedding_model = SentenceTransformer('all-MiniLM-L6-v2')
    print(f"✓ Loaded embedding model: all-MiniLM-L6-v2 (dim={embedding_model.get_sentence_embedding_dimension()})")

    chroma_client = chromadb.Client()
    # Cosine space ⇒ Chroma distance ≈ 1 - cosine_similarity (smaller is closer)
    try:
        chroma_client.delete_collection("medical_documents")
    except Exception:
        pass
    collection = chroma_client.create_collection(
        name="medical_documents",
        metadata={
            "hnsw:space": "cosine",
            "description": "Teaching corpus for RAG mechanism demo (not a verified literature DB)",
        },
    )

    documents = [doc["content"] for doc in MEDICAL_DOCUMENTS]
    ids = [doc["id"] for doc in MEDICAL_DOCUMENTS]
    metadatas = [{"title": doc["title"]} for doc in MEDICAL_DOCUMENTS]
    embeddings = embedding_model.encode(documents).tolist()
    collection.add(documents=documents, embeddings=embeddings, ids=ids, metadatas=metadatas)

    print(f"✓ Indexed {collection.count()} teaching documents (cosine space)")
else:
    print("⚠️ Skipping RAG setup - dependencies not available")


In [ ]:
def retrieve_context(query: str, n_results: int = 2) -> list:
    """Retrieve nearest teaching-corpus docs (cosine space)."""
    query_embedding = embedding_model.encode([query]).tolist()
    results = collection.query(
        query_embeddings=query_embedding,
        n_results=n_results,
        include=["documents", "metadatas", "distances"],
    )
    retrieved = []
    for i in range(len(results['documents'][0])):
        dist = float(results['distances'][0][i])  # cosine distance ≈ 1 - cos_sim
        retrieved.append({
            'document': results['documents'][0][i],
            'title': results['metadatas'][0][i]['title'],
            'distance': dist,
            'cosine_similarity': 1.0 - dist,
        })
    return retrieved

if RAG_AVAILABLE:
    test_query = "What loss function should I use for tumor segmentation?"
    print("=" * 60)
    print("RETRIEVAL TEST (teaching corpus; cosine space)")
    print("=" * 60)
    print(f"\nQuery: {test_query}\n")
    retrieved_docs = retrieve_context(test_query, n_results=2)
    for i, doc in enumerate(retrieved_docs):
        print(f"--- Retrieved Document {i+1} ---")
        print(f"Title: {doc['title']}")
        print(f"Cosine similarity: {doc['cosine_similarity']:.3f}  (distance={doc['distance']:.3f})")
        print(f"Content: {doc['document'][:200]}...")
        print()


### RAG-Augmented Question Answering

Now we combine retrieval with generation. The key insight is that we **prepend retrieved context** to the user's query, giving the LLM grounded information to work with.

**RAG Prompt Template:**
```
Based on the following context, answer the question.

Context:
{retrieved_documents}

Question: {user_query}

Answer:
```

This approach:
- Grounds the response in actual documents
- Enables source attribution ("According to document X...")
- Reduces hallucination by providing factual anchors

Generation uses the same **instruct model** loaded in Section 6 (`SmolLM2`), not the BART summarizer. Retrieval remains ChromaDB + MiniLM embeddings. In production you would typically use a larger local or private model (Llama, Mistral, BioMistral, etc.).


In [20]:
def rag_query(question: str, n_context: int = 2) -> dict:
    """
    Perform RAG: retrieve context and generate a grounded answer.

    Retrieval: ChromaDB + sentence-transformer embeddings
    Generation: instruct model from Section 6 (chat_generate)
    """
    # Step 1: Retrieve relevant documents
    retrieved = retrieve_context(question, n_results=n_context)

    # Step 2: Format context for the prompt
    context_text = "\n\n".join(
        [f"[Source: {doc['title']}]\n{doc['document']}" for doc in retrieved]
    )

    # Step 3: Create RAG prompt
    rag_prompt = f"""Based on the following context, answer the question concisely.
If the context is insufficient, say what is missing. Cite sources by title.

Context:
{context_text}

Question: {question}

Answer:"""

    # Step 4: Generate with the shared instruct model (fallback message if unavailable)
    if GENERATOR_AVAILABLE:
        answer = chat_generate(rag_prompt, max_new_tokens=180)
    else:
        # Fallback: return top retrieved snippet (still demonstrates grounding)
        answer = (
            "[Generator unavailable — showing top retrieved snippet]\n"
            + retrieved[0]["document"][:300]
            if retrieved
            else "[No retrieved context]"
        )

    return {
        "question": question,
        "answer": answer,
        "sources": [doc["title"] for doc in retrieved],
        "context": retrieved,
    }


# Test RAG wiring
if RAG_AVAILABLE:
    print("Testing RAG pipeline (retrieval + instruct generation)...")
    if not GENERATOR_AVAILABLE:
        print("⚠️ Instruct generator unavailable — answers will use retrieved snippets.")
else:
    print("⚠️ RAG not available - install chromadb / sentence-transformers to run this section")


Testing RAG pipeline (retrieval + instruct generation)...


In [21]:
# Demonstrate RAG with medical questions
if RAG_AVAILABLE:
    test_questions = [
        "What is the Dice loss and when should I use it?",
        "How can I reduce hallucination in medical LLMs?",
        "What is the U-Net architecture used for?"
    ]
    
    print("=" * 60)
    print("RAG-AUGMENTED QUESTION ANSWERING")
    print("=" * 60)
    
    for question in test_questions:
        print(f"\n{'─' * 50}")
        result = rag_query(question)
        
        print(f"❓ Question: {result['question']}\n")
        print(f"📚 Sources: {', '.join(result['sources'])}\n")
        print(f"💡 Answer: {result['answer']}")
    
    print(f"\n{'─' * 50}")
    print("\n✓ RAG provides grounded answers with source attribution")

[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


RAG-AUGMENTED QUESTION ANSWERING

──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ Question: What is the Dice loss and when should I use it?

📚 Sources: Dice Loss for Imbalanced Segmentation, BraTS Challenge and Brain Tumor Segmentation

💡 Answer: The Dice loss is a loss function used in medical image segmentation to optimize the overlap between predicted and ground truth regions. It is particularly effective for imbalanced datasets, such as brain tumor segmentation, where the region of interest (ROI) occupies a small fraction of the total volume. The Dice loss is defined as:

L_dice = 1 - (2 * sum(p * g) + epsilon) / (sum(p) + sum(g) + epsilon)

where p is the predicted probability and g is the ground truth. The epsilon term ensures numerical stability. Combining Dice loss with cross-entropy often provides more stable training than using either loss alone.

──────────────────────────────────────────────────


[transformers] Both `max_new_tokens` (=180) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


❓ Question: How can I reduce hallucination in medical LLMs?

📚 Sources: Hallucination Mitigation in Medical LLMs, Dice Loss for Imbalanced Segmentation

💡 Answer: To reduce hallucination in medical LLMs, you can use RAG to ground responses in verified documents, Chain-of-thought prompting to make reasoning explicit, Reinforcement learning from clinical feedback, and Calibrated confidence scores to flag uncertain outputs for human review. These strategies are particularly effective for imbalanced datasets.

──────────────────────────────────────────────────
❓ Question: What is the U-Net architecture used for?

📚 Sources: U-Net Architecture for Medical Image Segmentation, Dice Loss for Imbalanced Segmentation

💡 Answer: The U-Net architecture is used for biomedical image segmentation, particularly for tasks where precise boundary delineation is critical.

──────────────────────────────────────────────────

✓ RAG provides grounded answers with source attribution


### RAG vs. Base LLM: Key Differences

| Aspect | Base LLM | RAG-Augmented LLM |
|--------|----------|-------------------|
| **Knowledge source** | Parametric memory (training data) | Retrieved documents + parametric |
| **Hallucination risk** | Higher (relies on "memory") | Lower (grounded in documents) |
| **Source attribution** | ❌ Cannot cite sources | ✅ Can cite retrieved documents |
| **Knowledge updates** | Requires retraining | Add documents to index |
| **Domain specificity** | General knowledge | Can index domain-specific corpus |
| **Latency** | Single forward pass | Retrieval + generation |

### Where production RAG goes next

This demo used a **teaching corpus**. Production systems typically index:

- Institutional guidelines / protocols
- Peer-reviewed literature or curated evidence bases
- Verified pharmacology / formulary sources
- (With strict privacy controls) approved EHR-derived corpora

### Scaling RAG for Production

For production deployments, consider:

1. **Persistent vector stores**: Use Chroma with SQLite backend, or FAISS for larger corpora
2. **Chunking strategies**: Semantic chunking preserves meaning better than fixed-size splits
3. **Hybrid search**: Combine dense (embedding) and sparse (BM25) retrieval
4. **Reranking**: Use cross-encoders to rerank retrieved documents
5. **Evaluation**: Measure retrieval precision, answer relevance, and faithfulness

---


### Chain-of-Thought Prompting (Hallucination Mitigation)

Besides RAG, **chain-of-thought (CoT)** prompting asks the model to show intermediate reasoning before the final answer. This does not eliminate hallucinations, but it makes the reasoning path inspectable—and often improves answers on multi-step questions.

In this teaching demo we compare a direct answer vs. an explicit “think step by step” prompt using the same instruct model. Clinical CoT still requires expert review; never treat model reasoning as a diagnostic proof.


In [22]:
# Chain-of-thought vs direct prompting (same model, same question)
cot_question = (
    "A brain-tumor segmentation model reports whole-tumor Dice = 0.89 on a held-out test set. "
    "Does this alone demonstrate clinical readiness for autonomous use? "
    "Give a short conclusion."
)

direct_prompt = f"Question: {cot_question}\nAnswer briefly:"
cot_prompt = f"""Think step by step about validation needs (data shift, failure modes, human oversight), then conclude.
Question: {cot_question}
Reasoning:
"""

print("=" * 60)
print("CHAIN-OF-THOUGHT vs DIRECT PROMPTING")
print("=" * 60)

if GENERATOR_AVAILABLE:
    print("\n--- Direct ---")
    print(chat_generate(direct_prompt, max_new_tokens=120))
    print("\n--- Chain-of-thought ---")
    print(chat_generate(cot_prompt, max_new_tokens=220))
    print(
        "\n✓ CoT exposes intermediate claims you can challenge; "
        "pair with RAG when facts must be grounded in sources."
    )
else:
    print("⚠️ Generator unavailable — skipped CoT demo.")

# Link to extractive QA confidence (Section 7): span scores are useful flags,
# but they are not well-calibrated clinical probabilities.
print(
    "\nNote: extractive QA `score` values (Section 7) can flag low-confidence spans, "
    "but they are not calibrated clinical probabilities—use them only as review cues."
)


[transformers] Both `max_new_tokens` (=120) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


CHAIN-OF-THOUGHT vs DIRECT PROMPTING

--- Direct ---


[transformers] Both `max_new_tokens` (=220) and `max_length`(=20) seem to have been set. `max_new_tokens` will take precedence. Please refer to the documentation for more information. (https://huggingface.co/docs/transformers/main/en/main_classes/text_generation)


Yes, this alone demonstrates clinical readiness for autonomous use. The brain-tumor segmentation model's Dice score of 0.89 indicates that the model is able to accurately segment whole-tumor regions, which is a crucial step in the development of autonomous systems.

--- Chain-of-thought ---
Step 1: Identify the validation needs
Validation is a crucial step in the development of any AI model. It involves testing the model on unseen data to ensure it generalizes well to new situations. In this case, the validation needs are:

- Whole-tumor Dice = 0.89 on a held-out test set

Step 2: Analyze the validation results
The model's whole-tumor Dice score of 0.89 indicates that the model is performing well on the test set. However, the fact that the model's performance is not perfect (Dice = 0.89) suggests that there are still some areas for improvement.

Step 3: Evaluate the model's performance in real-world scenarios
To demonstrate clinical readiness for autonomous use, the model should be abl

## 9. Best Practices and Limitations

### Clinical Responsibility and AI Governance

The integration of LLMs into medical workflows raises important questions about **responsibility, validation, and governance**. Unlike traditional software with deterministic outputs, LLMs produce probabilistic responses that require new frameworks for clinical deployment.

<details>
<summary>📖 <b>The Human-in-the-Loop Imperative</b></summary>

Current regulatory guidance (FDA, EU AI Act) emphasizes that AI in healthcare should **augment, not replace** clinical judgment. Key principles:

1. **Clinician retains responsibility**: The physician, not the AI, is accountable for decisions
2. **Transparency**: Patients should be informed when AI assists in their care
3. **Audit trails**: Document AI contributions for quality assurance and liability
4. **Continuous monitoring**: Track AI performance in deployment, not just at validation
5. **Fail-safe design**: Systems should degrade gracefully when AI is unavailable
</details>

---

### When to Use LLMs in Medical Research

| ✅ **Good use cases** | ❌ **Avoid for** |
|----------------------|------------------|
| Literature summarization and review | Clinical decision-making (without validation) |
| Drafting documentation and reports | Patient-facing applications (without oversight) |
| Code boilerplate generation | Processing Protected Health Information |
| Exploratory text analysis | Generating medical advice or diagnoses |
| Educational and training purposes | Autonomous treatment recommendations |
| Research hypothesis generation | Replacing expert clinical judgment |

---

### Key Limitations and Mitigations

| Limitation | Risk | Mitigation |
|------------|------|------------|
| **Hallucination** | Plausible but incorrect medical information | Cross-reference outputs; use **RAG**; require citations; inspect **CoT** traces |
| **Bias** | Perpetuate disparities in training data | Audit across demographic groups; use diverse training data |
| **Knowledge cutoff** | Missing recent guidelines, drugs, research | Use RAG with current databases; verify currency of recommendations |
| **Context limits** | Long documents may be truncated | Chunk documents; retrieve only relevant passages |
| **Output variability** | Same prompt → different answers | Fix seeds where possible; sample multiple outputs; human review |

This notebook demonstrated **RAG**, **CoT prompting**, and **output variability**. Strategies such as reinforcement learning from clinical feedback (e.g., UniRG) and formal confidence calibration are active research directions—discussed in the chapter—and are not re-implemented here.


In [23]:
# Demonstration of output variability
print("=" * 60)
print("OUTPUT VARIABILITY DEMONSTRATION")
print("=" * 60)

# Same input, multiple summaries with sampling
text = SAMPLE_ABSTRACTS[1]['abstract']

print("\nGenerating 3 summaries with sampling enabled:")
print("-" * 40)

for i in range(3):
    # Tokenize input
    inputs = summarizer_tokenizer(
        " ".join(text.split()),
        return_tensors="pt",
        max_length=1024,
        truncation=True
    ).to(device)
    
    # Generate with sampling for variability
    with torch.no_grad():
        summary_ids = summarizer_model.generate(
            inputs["input_ids"],
            max_length=60,
            min_length=20,
            do_sample=True,  # Enable sampling for variability
            temperature=0.7,
            num_beams=1  # Use sampling instead of beam search
        )
    
    summary = summarizer_tokenizer.decode(summary_ids[0], skip_special_tokens=True)
    print(f"\nSummary {i+1}: {summary}")

OUTPUT VARIABILITY DEMONSTRATION

Generating 3 summaries with sampling enabled:
----------------------------------------

Summary 1: We present a multimodal deep learning framework that combines radiomics features extracted from CT scans with genomic profiles and clinical data to predict treatment response in lung cancer patients. Using a cohort of 422 patients, our attention-based fusion model achieved an AUC of 0.82 compared

Summary 2: Multi-modal deep learning framework combines radiomics features from CT scans with genomic profiles and clinical data to predict treatment response in lung cancer patients. The attention mechanism revealed that imaging features dominated predictions for early-stage tumors while genomic features were more important for advanced disease.

Summary 3: Radiomics features extracted from CT scans and genomic profiles and clinical data are combined to predict treatment response in non-small cell lung cancer patients. The attention mechanism revealed that imag

## 10. Summary

In this notebook, we demonstrated:

1. **Text Summarization**: Condensing medical abstracts with BART
2. **Zero-Shot Classification**: Categorizing texts without task-specific training
3. **Biomedical NER**: Transformer token-classification for clinical/scientific entities
4. **Code Generation**: Drafting helpers with a small instruct model, then comparing to reviewed templates
5. **Extractive Question Answering**: Span extraction with DistilBERT/SQuAD
6. **Retrieval-Augmented Generation (RAG)**: ChromaDB retrieval + instruct-model grounded answers
7. **Hallucination mitigations**: RAG, chain-of-thought prompting, and output-variability checks

### Key Takeaways

- LLMs are powerful tools for **research assistance**, not autonomous decision-making
- Always **validate outputs** against authoritative sources
- **RAG reduces hallucination risk** when the index holds trustworthy sources; our demo shows the mechanism on a teaching corpus
- **CoT** makes reasoning inspectable; it does not replace evidence retrieval or expert review
- Be mindful of **privacy** when using cloud-based models; consider local deployment
- Use **domain-specific models** (e.g., biomedical NER, clinical LMs) when available

### Next Steps

- Scale RAG with persistent vector stores (Chroma with SQLite, FAISS, Pinecone)
- Explore hybrid search combining dense and sparse retrieval
- Fine-tune embedding models on your domain-specific data
- Implement agentic workflows for multi-step reasoning tasks
- Set up local LLM deployment for sensitive data processing


In [24]:
# Clean up GPU memory if needed
if torch.cuda.is_available():
    torch.cuda.empty_cache()
    print("GPU memory cleared.")

GPU memory cleared.
